In [1]:
import torch
import json
from tqdm.notebook import tqdm
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
import os
from torch.utils.data import DataLoader
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
# ! pip install ipywidgets --upgrade

In [3]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "outputs-7B/final_lora",
    device_map="auto",
    load_in_4bit=True
)
processor = Qwen2VLProcessor.from_pretrained("outputs-7B/final_lora")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/home/quylm/.conda/lib/python3.14/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


In [4]:
def get_answer(image_path, question, model, processor, max_new_tokens=48):
    # Tạo prompt đúng định dạng multimodal
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        images=[image_path],
        text=[prompt],
        return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

In [5]:
image_path = "test-images/000000000050.jpg"
question = "Mô tả thật chi tiết ảnh này cho tôi ? "
answer = get_answer(image_path, question,model,processor)
answer

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'trong siêu thị có người phụ nữ mặc áo vàng đang lựa rau , bên cạnh là một người phụ nữ mặc áo trắng đang đẩy xe đẩy'

In [6]:
def build_image_path(image_id, image_lookup, images_root="training-images"):
    filename = image_lookup.get(str(image_id), f"{int(image_id):012d}.jpg")
    return f"{images_root}/{filename}"

def load_vqa_split(json_path, images_root="training-images"):
    with open(json_path, "r", encoding="utf-8") as f:
        train_json = json.load(f)

    image_lookup = train_json["images"]
    raw_data = []
    missing_images = 0

    for ann in train_json["annotations"].values():
        image_path = build_image_path(ann["image_id"], image_lookup, images_root)
        if os.path.exists(image_path):
            raw_data.append(
                {
                    "image_id": ann["image_id"],
                    "question": ann["question"],
                    "answer": ann["answer"],
                    "image_path": image_path,
                }
            )
        else:
            missing_images += 1

    print(f"Kept {len(raw_data)} samples, skipped {missing_images} missing images")
    return raw_data
raw_data = load_vqa_split("vlsp2023_train_data.json", "training-images")
dev_raw_data = load_vqa_split("vlsp2023_dev_data.json", "dev-images")

Kept 30833 samples, skipped 0 missing images
Kept 3545 samples, skipped 0 missing images


In [7]:
student_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map="auto",
    load_in_4bit=True
)
student_processor = Qwen2VLProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [8]:
# Dataset và DataLoader (giả sử raw_data đã có sẵn)
class VQADistillDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __getitem__(self, idx):
        return self.data[idx]
    def __len__(self):
        return len(self.data)

train_dataset = VQADistillDataset(raw_data)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [9]:
# Sử dụng AdamW 8-bit (bitsandbytes) cho optimizer
from bitsandbytes.optim import AdamW8bit
optimizer = AdamW8bit(student_model.parameters(), lr=2e-5)

In [10]:

# Cấu hình LoRA giống như Unsloth
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)

# Áp dụng LoRA cho student model
student_model = get_peft_model(student_model, lora_config)

# In số tham số cần train (trainable) và tổng số tham số
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in student_model.parameters())
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Tỷ lệ trainable: {100 * trainable_params / total_params:.2f}%")

Trainable parameters: 18,464,768
Total parameters: 1,240,740,352
Tỷ lệ trainable: 1.49%


In [ ]:
# Training loop với LoRA, distillation (soft label: KL loss giữa logits student và teacher, fix shape vocab/token)
from tqdm.notebook import tqdm
import torch.nn.functional as F

num_epochs = 1
student_model.train()
optimizer.zero_grad()

for epoch in range(num_epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for step, batch in enumerate(pbar):
        image_path = batch["image_path"][0]
        question = batch["question"][0]
        gt_answer = batch["answer"][0]
        # 1. Sinh answer từ teacher (để nối vào prompt)
        with torch.no_grad():
            teacher_answer = get_answer(image_path, question, model, processor)
        # 2. Tạo prompt multimodal và nối answer
        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
        ]
        prompt = student_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        full_text = prompt + teacher_answer
        # Không dùng truncation để tránh cắt mất token multimodal
        inputs = student_processor(
            images=[image_path],
            text=[full_text],
            return_tensors="pt",
            padding="max_length",
            max_length=512
        ).to(student_model.device)
        # Mask toàn bộ token prompt, chỉ tính loss cho answer
        labels = inputs["input_ids"].clone()
        prompt_len = len(student_processor.tokenizer(prompt)["input_ids"])
        labels[0, :prompt_len] = -100
        # 3. Forward teacher (lấy logits)
        with torch.no_grad():
            teacher_outputs = model(**inputs)
            teacher_logits = teacher_outputs.logits.detach()
        # 4. Forward student (lấy logits)
        student_outputs = student_model(**inputs)
        student_logits = student_outputs.logits
        # 5. Tính soft label loss (KLDiv) chỉ cho phần answer, fix shape vocab/token
        mask = (labels != -100)
        t_logits = teacher_logits[mask]
        s_logits = student_logits[mask]
        # Đảm bảo vocab size khớp
        min_vocab = min(t_logits.shape[-1], s_logits.shape[-1])
        t_logits = t_logits[..., :min_vocab]
        s_logits = s_logits[..., :min_vocab]
        # Đảm bảo số token giống nhau
        min_len = min(t_logits.shape[0], s_logits.shape[0])
        t_logits = t_logits[:min_len, :]
        s_logits = s_logits[:min_len, :]
        loss = F.kl_div(
            F.log_softmax(s_logits, dim=-1),
            F.softmax(t_logits, dim=-1),
            reduction="batchmean"
        )
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        torch.cuda.empty_cache()
        pbar.set_postfix({"loss": loss.item()})


In [ ]:
# Lưu model và processor sau khi train vào outputs-distill
save_dir = "outputs-distill/final_lora"
os.makedirs(save_dir, exist_ok=True)
student_model.save_pretrained(save_dir)
student_processor.save_pretrained(save_dir)
print(f"Đã lưu student model và processor vào {save_dir}")

Đã lưu student model và processor vào outputs-distill/final_lora
